 # Designing an API Rate Limiter

An API reate limiter will throttle users based on the number of requests they are sending.

## Why Rate Limiting?
Rate limiting is all about traffic/load management. Clients can misbehave and trigger a retry storm, others might request the same information over and over again, increasing traffic and unnecessary load on the system. This can degrade the performance of the service, and the ability of the system to reliably handle incoming requests.  

Rate limiting helps to protect services against abusive behaviors targeting the application such as retry storms, Denial-of-service (DOS) attacks, brute-force password attempts, brute-force credit card transactions, etc.

Rate limiting saves the company money by eliminating service and infrastructure costs that would have otherwise been used to handle spamming.

Here are some scenarios to show the importance of Rate limiting our API/Services:

- **Prevents service degradation:** It reduces traffic spikes so that the service stays reliable for all users.

- **Misbehaving clients:** Sometimes, clients can overwhelm servers by sending large number of requests, either intentionally or unintentionally. 

- **Security:** Limiting the number of times a user should authenticate with a wrong password.

- **Preventing abusive and bad design practices:** Without API limits, developers of client apps might request the same info over and over again.

- **Revenue:**  Services can limit operations based on the tier of their customer's service and thus create a revenue model off the rate limiting. To go beyond the set limit, the user has to buy higher limits. 


## 1. Requirements and System Goals

#### Functional requirements
1. Apply a policy to a defined key, such as `(user_id, API operation)`, across all API servers handling that key.
2. Count admissions, not successful backend responses. An admitted request consumes quota even if the backend later fails; a rejected attempt does not. A retry is a new attempt unless a separate idempotency contract says otherwise.
3. For an exact rolling policy with limit `L` and window `W`, admit a request at time `t` only if fewer than `L` prior admissions lie in `(t-W, t]`. An admission exactly at `t-W` has expired. Distinct attempts at the same timestamp count separately.

#### Non-Functional requirements
1. Define the failure policy: fail-closed rejects requests when the admission authority is unavailable; fail-open preserves access but relaxes the limit. Independent local counters do not preserve a hard global limit.
2. Keep the decision path bounded and measure its latency under the expected load, including hot keys. The examples below assume nondecreasing authoritative time; production clock correction, restart, and failover behavior must preserve the chosen accounting contract.

## 2. Throttling Types

* ***Hard Throttling*** – Number of API requests cannot exceed the throttle limit.

* ***Soft Throttling*** – Set the API request limit to exceed by some percentage. E.g, if the rate-limit = 100 messages/minute, and 10% exceed-limit, our rate limiter will allow up to 110 messages per minute.

* ***Dynamic Throttling (Priority throttling)*** – The number of requests can exceed the limit if the system has some free resources available. The system can progressively throttle requests based on some predefined priority. 

## 3. Algorithms used for Rate Limiting

#### Fixed Window Algorithm
Use aligned, half-open buckets: for a one-second window, `[0,1000)`, `[1000,2000)`, and so on, with time in milliseconds. Each bucket permits at most `L` admissions. This is a different contract from the exact rolling policy: requests on opposite sides of a boundary use different quotas.

In the diagram, with `L = 2` per second, the fixed-window policy rejects only `m5`.

![](images/fixed_rolling_window.svg)

#### Rolling Window Algorithm
Look backward from every request, not forward from the first request. With `L = 2` and `W = 1000 ms`, admissions at `300 ms` and `400 ms` cause `m3` and `m4` in the diagram to be rejected. Those admissions expire at `1300 ms` and `1400 ms`, respectively, so later requests can succeed without an aligned reset.




## 4. High Level Design

Once a new request arrives, the Web server first asks the Rate Limiter to decide if it will be served or throttled. If the request is not throttled, then it's passed to the API servers. 

![](images/rate_limiter_high_level_design.png)


## 5. Basic System Design and Algorithm

For a fixed-window limit of `L = 3` admissions per second, store a bucket ID and admission count for each policy key. At time `t` in milliseconds:

1. Calculate `bucket_id = floor(t / 1000)`.
2. If the key is absent or its stored bucket differs, use this bucket with count zero.
3. Reject if `count >= 3`; otherwise increment the count and admit.

The read, bucket change, comparison, and increment must be one atomic decision.

#### Problems with this Fixed Window Algorithm
1. A fixed-window policy permits a boundary burst. Three attempts at `999 ms` followed by three at `1000 ms` are all admitted: six admissions span just one millisecond. Each aligned bucket still satisfies its limit. Use an exact sliding window if that burst violates the requirement.

![](images/fixed_window_problem.svg)

The diagram illustrates the same boundary effect using minute-long buckets; the example above uses seconds.

2. Separate reads and writes race. At count two, two concurrent requests can both see a free slot and both admit, exceeding the limit of three.

![](images/fixed_window_atomicity.svg)


#### Solutions
Use a bounded server-side operation at the authority for the policy key. For example, a [Redis Lua script](https://redis.io/docs/latest/develop/programmability/eval-intro/) can perform the read/check/update without interleaving other commands. Keep all keys for that decision on the same supported shard/slot.

This removes the read/write race, not every failure mode. Scripts block other work while running; bound their work. Atomicity here means no interleaving, not rollback of earlier writes if a script errors. It does not by itself protect acknowledged state across failover or authorize multiple independent primaries to admit for the same key.

&nbsp;

#### How much memory to store all the user data?
For a payload-only estimate, assume eight bytes each for a policy-key identifier, bucket ID, and counter:

```text
24 bytes/key * 10,000,000 active keys = 240,000,000 bytes = 240 MB
```

All sizes here use decimal units (`kB = 1000 bytes`, `GB = 10^9 bytes`). These are logical payloads, not measured Redis allocations: key encoding, data structures, allocator overhead, expiry metadata, replication, and headroom are additional.

Memory fit does not establish throughput. One million simultaneously active keys at ten attempts/second require ten million admission decisions/second. Measure the actual operation mix and hot-key concentration before choosing shard count; adding API servers does not split a single key's authoritative decision.



## 6. Sliding Window Algorithm

Keep accepted admissions for each policy key in a sorted set. The score is the admission timestamp; the member identifies the distinct attempt. [Redis sorted-set members are unique](https://redis.io/docs/latest/commands/zadd/): using the timestamp as both score and member would collapse simultaneous admissions.

For `L = 3` and `W = 1000 ms`, execute these steps atomically at authoritative time `t`:

1. Remove admissions with timestamp `<= t-W`, leaving exactly `(t-W, t]`.
2. Reject if the remaining count is `>= 3`. Do not insert a rejected attempt.
3. Otherwise insert a distinct member with score `t` and admit.

Expire an inactive key no earlier than its last admission plus `W`. Expiry reclaims idle state; pruning and the count check determine eligibility.

The following expected trace includes simultaneous attempts, a rejection, and success exactly when earlier admissions expire:

| Attempt | Time (ms) | Prior admissions in `(t-1000, t]` | Decision |
|---|---:|---:|---|
| A | 100 | 0 | Admit |
| B | 100 | 1 | Admit |
| C | 100 | 2 | Admit |
| D | 100 | 3 | Reject |
| E | 1099 | 3 | Reject |
| F | 1100 | 0 | Admit |

#### Memory for storing user data?
For the memory comparison, use a nominal quota of 500 admissions/hour and ten million active keys. Assume an eight-byte policy-key identifier, eight-byte timestamps, and packed 16-byte distinct members:

```text
8 + 500 * (8 + 16) = 12,008 bytes/key
12,008 * 10,000,000 = 120,080,000,000 bytes = 120.08 GB
```

This is the maximum retained logical payload at that quota, before storage overhead and replicas. It is not a measured Redis footprint or evidence that the design cannot scale. Coarser counters save state by changing the precision of the decision.


## 7. Sliding Window + Counters
Use aligned minute buckets for a conservative approximation to a rolling-hour limit of 500 admissions. An additional limit of ten admissions per aligned minute is a separate product policy, not an assumption about what legitimate users need.

At time `t` in seconds, retain buckets whose end is strictly greater than `t-3600`. Count the entire oldest overlapping bucket because individual timestamps are unavailable. Before incrementing, reject if the retained sum is `>= 500` or the current minute's count is `>= 10`; otherwise increment the current bucket and admit. Pruning, both comparisons, and incrementing form one atomic decision.

Up to **61 buckets** are needed: the current minute plus the preceding buckets that overlap `(t-3600, t]`. Counting the oldest whole bucket can overcount by up to ten admissions under the separate minute cap, causing false rejections rather than exceeding the exact rolling limit.

For example, admit ten requests at `t=0` and 490 more across later minutes. At `t=3600`, the ten requests at zero have expired under the exact rolling policy, so a new request is eligible. The bucket approximation still counts their bucket and rejects at 500. At `t=3660`, that bucket is removed; with no intervening admissions, the retained count is 490 and a request is admitted.

Store buckets in a Redis hash. Explicitly delete obsolete bucket fields on active keys. A whole-key [EXPIRE](https://redis.io/docs/latest/commands/expire/) reclaims an idle hash; refreshing it does not prune old fields in a continuously active hash. A TTL of at least `W + one bucket width` after the last admission is sufficient for this conservative retention rule.

This is not an exact rolling implementation. Use it only when the memory saving justifies the stated false-rejection behavior; retrying next minute does not guarantee success while the hourly quota remains occupied.

#### How much memory to store all user data?

Using the same payload-only accounting, with eight bytes each for a bucket ID and counter:

```text
8 + 61 * (8 + 8) = 984 bytes/key
984 * 10,000,000 = 9,840,000,000 bytes = 9.84 GB
1 - 984 / 12,008 = about 91.8% less logical payload than the admission log
```

The additional minute cap changes burst behavior, and the bucket approximation changes precision. Neither this calculation nor an entry-count threshold predicts Redis's actual encoding or allocated memory; measure the chosen version and representation.


## 8. Data Sharding and Caching
We can shard by `UserID` to distribute user data across different partitions.

Consistent hashing can reduce key movement when membership changes; it does not provide replication or recover lost counters. Define how routing, state transfer, and failover preserve one admission authority per key. Moving a key without its unexpired state grants quota again.

&nbsp;

## Caching
Independent write-back caches cannot enforce the hard shared limit: two API servers can each spend the same remaining quota before either writes back. Reconciliation cannot undo those admissions.

Cache policy configuration with an explicit update policy, but keep quota decisions at the authority. Local allowances require disjoint preallocated budgets and a safe ownership lifecycle, which this sketch does not implement. LRU eviction is suitable only for nonauthoritative caches; discarding an unexpired authoritative counter silently restores quota.


Sample response:



# 9. Throttling Response
We can return a 429 status code: Too Many Requests whenever the user exceeds the rate limit.

| Header Name          | Description                                                                    |
| :------------------- | :---------------------------------------------------------------------------- |
| RateLimit-Exceeded   | The specific limit that has been exceeded.                                     |
| Retry-After          | The number of seconds that the client should wait before retrying a request.   |


```
HTTP/1.1 429 Too Many Requests
Transfer-Encoding: chunked
Retry-After: 5
request-id: ff4751b7-d289-01a0-a6dd-9b3541c077fe
RateLimit-Exceeded: 60
Cache-Control: private
Content-Type: application/json;charset=utf-8
```
```json
{
  "error": {
    "code": "ClientThrottled",
    "message": "Client application is over its resource limit."
  }
}